<a href="https://colab.research.google.com/github/NicKylis/SveltNet/blob/development/exp_pytorch_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score
import os
import requests
import zipfile
import io
import gzip
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

# Create directory and download EMNIST dataset
os.makedirs('res', exist_ok=True)
data = requests.get('https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip')
files = zipfile.ZipFile(io.BytesIO(data.content))
files.extractall('res')

# Validation split & batch size
validation_split = 0.3
batch_size = 512
num_of_classes = 10  # EMNIST has 62 characters / classes (0-9, A-Z, a-z)

# Function to read MNIST images
def read_MNIST_images(filename):
    with gzip.open(filename, 'rb') as file:
        images = np.frombuffer(file.read(), np.uint8, offset=16)
    return images.reshape(-1, 28, 28).astype("float32") / 255.0

# Function to read MNIST labels
def read_MNIST_labels(filename):
    with gzip.open(filename, 'rb') as file:
        labels = np.frombuffer(file.read(), np.uint8, offset=8)
    return labels

# Load data
x_train = read_MNIST_images('res/gzip/emnist-digits-train-images-idx3-ubyte.gz')
y_train = read_MNIST_labels('res/gzip/emnist-digits-train-labels-idx1-ubyte.gz')
x_test = read_MNIST_images('res/gzip/emnist-digits-test-images-idx3-ubyte.gz')
y_test = read_MNIST_labels('res/gzip/emnist-digits-test-labels-idx1-ubyte.gz')

# Split into train & validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=validation_split, random_state=42)

# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)  # (N, 1, H, W)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # Class indices, not one-hot

x_val_tensor = torch.tensor(x_val, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# PyTorch Dataset class
class MNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Create datasets
train_dataset = MNISTDataset(x_train_tensor, y_train_tensor)
val_dataset = MNISTDataset(x_val_tensor, y_val_tensor)
test_dataset = MNISTDataset(x_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [2]:
# Define the CNN model with corrected channels
class TAAF(nn.Module):
    def __init__(self, in_features):
        super(TAAF, self).__init__()
        # Initialize learnable parameters for scaling and shifting
        self.alpha = nn.Parameter(torch.ones(1))  # Scale parameter (learnable)
        self.beta = nn.Parameter(torch.zeros(1))  # Shift parameter (learnable)

    def forward(self, x):
        # TAAF function can be some form of scaled, shifted non-linearity
        # For example, a simple TAAF could be a scaled and shifted ReLU
        return self.alpha * F.relu(x + self.beta)

class CNNModel(nn.Module):
    def __init__(self, num_of_classes=10):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2, bias=False)  # 1 input channel
        self.bn1 = nn.BatchNorm2d(16)

        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(16)

        self.dwconv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)  # 16 -> 32
        self.bn4 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.15)

        self.dwconv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn5 = nn.BatchNorm2d(32)

        self.dwconv6 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn6 = nn.BatchNorm2d(32)

        self.conv7 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)  # 32 -> 64
        self.bn7 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.2)

        self.dwconv8 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn8 = nn.BatchNorm2d(64)

        self.dwconv9 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn9 = nn.BatchNorm2d(64)

        self.conv10 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn10 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.2)

        self.dwconv11 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn11 = nn.BatchNorm2d(64)

        self.dwconv12 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn12 = nn.BatchNorm2d(64)

        self.conv13 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn13 = nn.BatchNorm2d(64)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop4 = nn.Dropout(0.2)

        self.dwconv14 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn14 = nn.BatchNorm2d(64)

        self.dwconv15 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn15 = nn.BatchNorm2d(64)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.drop5 = nn.Dropout(0.25)
        self.fc = nn.Linear(64, num_of_classes)

        self.taaf = TAAF(1024)

    # def forward(self, x):
    #     x = F.relu(self.bn1(self.conv1(x)))
    #     x = F.relu(self.bn2(self.dwconv2(x)))
    #     x = F.relu(self.bn3(self.dwconv3(x)))

    #     x = F.relu(self.bn4(self.conv4(x)))
    #     x = self.pool1(x)
    #     x = self.drop1(x)

    #     x = F.relu(self.bn5(self.dwconv5(x)))
    #     x = F.relu(self.bn6(self.dwconv6(x)))

    #     x = F.relu(self.bn7(self.conv7(x)))
    #     x = self.pool2(x)
    #     x = self.drop2(x)

    #     x = F.relu(self.bn8(self.dwconv8(x)))
    #     x = F.relu(self.bn9(self.dwconv9(x)))

    #     x = F.relu(self.bn10(self.conv10(x)))
    #     x = self.pool3(x)
    #     x = self.drop3(x)

    #     x = F.relu(self.bn11(self.dwconv11(x)))
    #     x = F.relu(self.bn12(self.dwconv12(x)))

    #     x = F.relu(self.bn13(self.conv13(x)))
    #     x = self.pool4(x)
    #     x = self.drop4(x)

    #     x = F.relu(self.bn14(self.dwconv14(x)))
    #     x = F.relu(self.bn15(self.dwconv15(x)))

    #     x = self.global_pool(x)
    #     x = torch.flatten(x, 1)
    #     x = self.drop5(x)
    #     x = self.fc(x)
    #     return x

    def forward(self, x):
        x = self.taaf(self.bn1(self.conv1(x)))
        x = self.taaf(self.bn2(self.dwconv2(x)))
        x = self.taaf(self.bn3(self.dwconv3(x)))

        x = self.taaf(self.bn4(self.conv4(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.taaf(self.bn5(self.dwconv5(x)))
        x = self.taaf(self.bn6(self.dwconv6(x)))

        x = self.taaf(self.bn7(self.conv7(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.taaf(self.bn8(self.dwconv8(x)))
        x = self.taaf(self.bn9(self.dwconv9(x)))

        x = self.taaf(self.bn10(self.conv10(x)))
        x = self.pool3(x)
        x = self.drop3(x)

        x = self.taaf(self.bn11(self.dwconv11(x)))
        x = self.taaf(self.bn12(self.dwconv12(x)))

        x = self.taaf(self.bn13(self.conv13(x)))
        x = self.pool4(x)
        x = self.drop4(x)

        x = self.taaf(self.bn14(self.dwconv14(x)))
        x = self.taaf(self.bn15(self.dwconv15(x)))

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.drop5(x)
        x = self.fc(x)
        return x

# Initialize model
model = CNNModel(num_of_classes=num_of_classes)

# Device management
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=2.5e-5)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_weights = model.state_dict() if self.restore_best_weights else None
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                if self.restore_best_weights and self.best_model_weights is not None:
                    model.load_state_dict(self.best_model_weights)
                return True
        return False

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

In [3]:


# Training Loop
# num_epochs = 30
# for epoch in range(num_epochs):
#     model.train()
#     train_loss = 0.0
#     for batch in train_loader:  # Use DataLoader, not Dataset
#         inputs, labels = batch
#         inputs, labels = inputs.to(device), labels.to(device)  # Move to device
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = torch.nn.CrossEntropyLoss()(outputs, labels)  # Expects class indices
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item() * inputs.size(0)  # Weighted by batch size

#     train_loss /= len(train_loader.dataset)  # Average over dataset size

#     # Validation
#     model.eval()
#     val_loss = 0.0
#     with torch.no_grad():
#         for batch in val_loader:  # Use DataLoader, not Dataset
#             inputs, labels = batch
#             inputs, labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)
#             loss = torch.nn.CrossEntropyLoss()(outputs, labels)
#             val_loss += loss.item() * inputs.size(0)

#     val_loss /= len(val_loader.dataset)  # Average over dataset size

#     # Reduce LR on Plateau
#     scheduler.step(val_loss)

#     # Early Stopping Check
#     if early_stopping(val_loss, model):
#         break

#     print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

num_epochs = 30
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = torch.nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()

        # Compute training loss
        train_loss += loss.item() * inputs.size(0)

        # Compute training accuracy
        _, predicted = torch.max(outputs, 1)  # Get the index of the max logit (predicted class)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= len(train_loader.dataset)
    train_accuracy = 100. * train_correct / train_total

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = torch.nn.CrossEntropyLoss()(outputs, labels)

            # Compute validation loss
            val_loss += loss.item() * inputs.size(0)

            # Compute validation accuracy
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_accuracy = 100. * val_correct / val_total

    # Reduce LR on Plateau
    scheduler.step(val_loss)

    # Early Stopping Check
    if early_stopping(val_loss, model):
        break

    # Print both loss and accuracy
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

Epoch 1/30 - Train Loss: 0.3338, Train Acc: 91.06%, Val Loss: 0.0552, Val Acc: 98.47%
Epoch 2/30 - Train Loss: 0.0583, Train Acc: 98.57%, Val Loss: 0.0383, Val Acc: 99.00%
Epoch 3/30 - Train Loss: 0.0433, Train Acc: 98.90%, Val Loss: 0.0320, Val Acc: 99.18%
Epoch 4/30 - Train Loss: 0.0375, Train Acc: 99.09%, Val Loss: 0.0270, Val Acc: 99.28%
Epoch 5/30 - Train Loss: 0.0349, Train Acc: 99.16%, Val Loss: 0.0268, Val Acc: 99.31%
Epoch 6/30 - Train Loss: 0.0315, Train Acc: 99.25%, Val Loss: 0.0258, Val Acc: 99.32%
Epoch 7/30 - Train Loss: 0.0295, Train Acc: 99.25%, Val Loss: 0.0213, Val Acc: 99.45%
Epoch 8/30 - Train Loss: 0.0262, Train Acc: 99.33%, Val Loss: 0.0211, Val Acc: 99.47%
Epoch 9/30 - Train Loss: 0.0251, Train Acc: 99.37%, Val Loss: 0.0231, Val Acc: 99.40%
Epoch 10/30 - Train Loss: 0.0232, Train Acc: 99.42%, Val Loss: 0.0232, Val Acc: 99.42%
Epoch 11/30 - Train Loss: 0.0234, Train Acc: 99.42%, Val Loss: 0.0193, Val Acc: 99.51%
Epoch 12/30 - Train Loss: 0.0219, Train Acc: 99.48%,

In [4]:
# Test Phase (after training)
# model.eval()
# test_correct = 0
# test_total = 0

# with torch.no_grad():
#     for batch in test_loader:
#         inputs, labels = batch
#         inputs, labels = inputs.to(device), labels.to(device)
#         outputs = model(inputs)
#         _, predicted = torch.max(outputs, 1)
#         test_total += labels.size(0)
#         test_correct += (predicted == labels).sum().item()

# test_accuracy = 100. * test_correct / test_total
# print(f"Test Accuracy: {test_accuracy:.10f}%")
model.eval()
all_preds = []
all_labels = []
test_loss = 0.0
test_correct = 0
test_total = 0

# Define the loss function
criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        # Loss
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Accuracy
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

# Convert lists to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Average test loss
test_loss /= len(test_loader.dataset)

# Accuracy
test_accuracy = 100. * test_correct / test_total

# Precision and recall
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.10f}%")
print(f"Test Precision (Weighted): {precision:.4f}")
print(f"Test Recall (Weighted): {recall:.4f}")

Test Loss: 0.0139
Test Accuracy: 99.6950000000%
Test Precision (Weighted): 0.9970
Test Recall (Weighted): 0.9970


In [5]:
def print_parameters_breakdown(model):
    print("Layer-wise Parameter Breakdown:")
    print("-" * 40)
    total_params = 0
    for name, param in model.named_parameters():
        num_params = param.numel()
        total_params += num_params
        print(f"{name}: {num_params:,} parameters")
    print("-" * 40)
    print(f"Total Parameters: {total_params:,}")

# Print detailed breakdown
print_parameters_breakdown(model)

Layer-wise Parameter Breakdown:
----------------------------------------
conv1.weight: 400 parameters
bn1.weight: 16 parameters
bn1.bias: 16 parameters
dwconv2.weight: 144 parameters
bn2.weight: 16 parameters
bn2.bias: 16 parameters
dwconv3.weight: 144 parameters
bn3.weight: 16 parameters
bn3.bias: 16 parameters
conv4.weight: 4,608 parameters
bn4.weight: 32 parameters
bn4.bias: 32 parameters
dwconv5.weight: 288 parameters
bn5.weight: 32 parameters
bn5.bias: 32 parameters
dwconv6.weight: 288 parameters
bn6.weight: 32 parameters
bn6.bias: 32 parameters
conv7.weight: 18,432 parameters
bn7.weight: 64 parameters
bn7.bias: 64 parameters
dwconv8.weight: 576 parameters
bn8.weight: 64 parameters
bn8.bias: 64 parameters
dwconv9.weight: 576 parameters
bn9.weight: 64 parameters
bn9.bias: 64 parameters
conv10.weight: 36,864 parameters
bn10.weight: 64 parameters
bn10.bias: 64 parameters
dwconv11.weight: 576 parameters
bn11.weight: 64 parameters
bn11.bias: 64 parameters
dwconv12.weight: 576 parameter

In [8]:
torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }, 'model_weights.pth')